# Silver — pipeline declarativo (etapas 2.1 a 2.5)

Pipeline Lakeflow Spark Declarative Pipelines que lê as tabelas de `olist.bronze` (tudo `string`, com duplicatas) e entrega, para cada entidade:

- `olist.silver.<entidade>`: tipos corretos e, nas transacionais, uma linha por chave;
- `olist.quality.<entidade>_quarentena`: os registros reprovados, com o motivo.

**Entidades:** `orders` (2.1) · `order_items`, `order_payments`, `order_reviews`, `order_status_events`, `products`, `geolocation`, `category_translation` (2.2) · `customers`, `sellers` (2.5).

**Etapa 2.5:** `customers` e `sellers` são feeds de mudança (CDC): cada linha traz `operation` (INSERT, UPDATE, DELETE), `change_ts` e `change_sequence` (1, 2, 3: a versão da linha na chave). A silver guarda **uma linha por chave, a de maior `change_sequence`**, e mantém `operation` e `change_ts`. O histórico de versões (SCD tipo 2) e o tratamento das exclusões ficam para a Fase 3.

**Etapa 2.3:** regras de domínio (valor não negativo, `review_score` de 1 a 5, status e tipo de pagamento conhecidos), regras de aviso que só medem (`payment_gateway`) e correção determinística de formato (`price` com vírgula decimal), marcada em `_dq_correcao`.

**Este notebook é a fonte de um pipeline, não um notebook comum.** Ele não roda célula a célula: o pipeline lê todas as células, monta o grafo e executa na ordem certa. Use *Dry run* para validar e *Run pipeline* para executar (roteiro na última célula).

**Configuração do pipeline**

| Chave | Padrão | Uso |
|---|---|---|
| `olist.catalogo` | `olist` | catálogo de origem e destino; permite o prefixo `dev_` na Fase 7 (desenho S8) |

**Fluxo de cada entidade** (o mesmo para todas, escrito uma vez só na função `registrar_entidade`)

```
bronze.<e> ─► <e>_tipado (view: tipos + correções) ─┬─► <e>_validos (view + expectations) ─► silver.<e>
                                                    └─► quality.<e>_quarentena (reprovados + motivo)
```

Desenho: `docs/design/fase-2-silver-qualidade.md` (decisões S1 a S5 e S8).

## Configuração

Importa as bibliotecas do pipeline e lê o catálogo da configuração, sem caminho fixo no código (desenho S8). Nenhum dado é lido aqui.

In [0]:
# O QUE FAZ: importa o módulo do pipeline, as funções de coluna do Spark e o dataclass, e lê o
#   catálogo da configuração do pipeline. Nenhum dado é lido aqui.
#
# IMPORTAÇÕES:
# - from pyspark import pipelines as dp: módulo das Spark Declarative Pipelines. Fornece os
#   decoradores (@dp.view, @dp.table, @dp.expect_all_or_drop) e as funções create_streaming_table
#   e create_auto_cdc_flow. Só existe dentro de um pipeline.
# - Column e DataFrame: usados apenas nas anotações de tipo das funções.
# - pyspark.sql.functions as F: funções nativas de coluna (col, expr, when, coalesce...).
# - dataclass (biblioteca padrão): gera uma classe só de dados, usada para descrever cada entidade.
#
# FUNÇÕES NATIVAS:
# - spark.conf.get(chave, padrão): lê um valor da configuração do pipeline (Settings →
#   Configuration). Sem a chave definida, devolve o padrão ("olist").
from dataclasses import dataclass, field

from pyspark import pipelines as dp
from pyspark.sql import Column, DataFrame
from pyspark.sql import functions as F

CATALOG = spark.conf.get("olist.catalogo", "olist")

## Funções puras de transformação

Convenção do projeto: a transformação é uma função **DataFrame → DataFrame** (ou devolve um texto/coluna), sem ler nem gravar nada, testável sem cluster.

- `tipar`: converte as colunas de `string` para o tipo de negócio **sem falhar** (`try_cast`: valor inválido vira nulo) e guarda o texto original em `<coluna>_bruto`, para a quarentena mostrar o que chegou.
- `montar_regras`: gera as regras de qualidade de uma entidade a partir da sua descrição. Cada regra tem **nome** e uma condição SQL verdadeira para o registro *válido*.
- `corrigir_formato`: **etapa 2.3.** Quando a conversão falhou, mas o texto tem um formato conhecido e a correção é determinística (por exemplo, `59,90` com vírgula decimal), converte o valor corrigido e registra o nome da correção em `_dq_correcao`. Formato desconhecido continua nulo e vai para a quarentena.
- `condicao_valida`, `motivos_reprovacao`, `isolar_reprovados`: transformam as regras em filtro e em quarentena com motivo.

O mesmo dicionário de regras alimenta as expectations (que descartam o registro da silver) e a quarentena (que o guarda com o motivo), então os dois lados nunca divergem.

**Regra de conversão (desenho S4):** uma coluna só reprova quando *havia valor e a conversão virou nulo*. Um nulo que já vinha nulo da origem (por exemplo, o peso de um produto sem cadastro) é válido, a menos que a coluna seja obrigatória.

In [0]:
# O QUE FAZ: declara as funções puras de transformação. Nada é executado aqui: elas só são
#   chamadas por registrar_entidade, mais adiante.
#
# FUNÇÕES DECLARADAS:
# - tipar(df, tipos): para cada coluna de "tipos" ({coluna: tipo SQL}), guarda o texto original em
#   <coluna>_bruto e substitui a coluna pelo valor convertido (nulo se não converter).
# - corrigir_formato(df, correcoes, tipos): para cada correção, onde a coluna ficou nula e o texto
#   original combina com o formato reconhecido, converte o texto corrigido e preenche
#   _dq_correcao com o nome da correção. Sem correções, devolve o DataFrame como está.
# - montar_regras(chaves, obrigatorias, tipos): devolve o dicionário {nome da regra: condição SQL}.
#   Colunas da chave e obrigatórias nunca podem ser nulas; toda coluna tipada precisa ser
#   conversível; a linha não pode ser corrompida (ADR 0004) nem ter dado fora do schema.
# - condicao_valida(regras): UMA coluna booleana, verdadeira só se todas as regras forem
#   verdadeiras. Resultado nulo de uma regra conta como reprovado (coalesce com False).
# - motivos_reprovacao(regras): coluna de texto com os nomes das regras violadas, separados por vírgula.
# - isolar_reprovados(df, regras): mantém só os reprovados e acrescenta motivo e _quarentena_em.
#
# FUNÇÕES NATIVAS:
# - DataFrame.withColumns({nome: expressão}): cria ou substitui várias colunas de uma vez; todas
#   as expressões enxergam as colunas originais (por isso o *_bruto guarda o texto antes da troca).
# - F.col(nome): referência a uma coluna.
# - coluna.isNull() / coluna.rlike(regex): teste de nulo / teste de expressão regular.
# - F.when(cond, a).otherwise(b): a se a condição é verdadeira, senão b.
# - regexp_replace(texto, padrão, novo) (SQL): troca o padrão pelo novo texto; aqui troca a vírgula
#   decimal por ponto antes da conversão.
# - F.nullif(a, b): devolve nulo quando a é igual a b; usado para que "sem correção" fique nulo,
#   e não texto vazio.
# - F.expr("try_cast(x AS TIPO)"): expressão SQL. try_cast devolve NULL quando a conversão não é
#   possível, em vez de derrubar o pipeline (desenho S4).
# - F.lit(valor): coluna com valor constante.
# - F.coalesce(a, b): primeiro valor não nulo; aqui transforma "nulo" em False.
# - F.when(condição, valor): valor quando a condição é verdadeira, senão nulo.
# - F.concat_ws(sep, ...): junta textos com separador e ignora os nulos.
# - ~coluna: negação lógica (NOT).
# - DataFrame.where(condição): filtra linhas.
# - F.current_timestamp(): momento em que a linha foi processada.
# - dict.items() / dict.values(): percorre pares (chave, valor) ou só os valores de um dicionário.
def tipar(df: DataFrame, tipos: dict[str, str]) -> DataFrame:
    """Converte os tipos sem falhar (texto inválido vira nulo) e guarda o original em *_bruto."""
    novas = {}
    for coluna, tipo in tipos.items():
        novas[f"{coluna}_bruto"] = F.col(coluna)
        novas[coluna] = F.expr(f"try_cast({coluna} AS {tipo})")
    return df.withColumns(novas)


def corrigir_formato(df: DataFrame, correcoes: tuple, tipos: dict[str, str]) -> DataFrame:
    """Converte formatos conhecidos que o try_cast rejeitou e marca a correção em _dq_correcao."""
    if not correcoes:
        return df
    novas = {}
    marcas = []
    for c in correcoes:
        aplicar = F.col(c.coluna).isNull() & F.col(f"{c.coluna}_bruto").rlike(c.padrao)
        corrigido = F.expr(
            f"try_cast(regexp_replace({c.coluna}_bruto, '{c.de}', '{c.para}') AS {tipos[c.coluna]})"
        )
        novas[c.coluna] = F.when(aplicar, corrigido).otherwise(F.col(c.coluna))
        marcas.append(F.when(aplicar, F.lit(c.nome)))
    # A marca entra no MESMO withColumns da troca: todas as expressões enxergam as colunas
    # originais. Num segundo passo, a coluna já estaria corrigida (não nula) e a marca sairia nula.
    novas["_dq_correcao"] = F.nullif(F.concat_ws(", ", *marcas), F.lit(""))
    return df.withColumns(novas)


def montar_regras(
    chaves: tuple[str, ...], obrigatorias: tuple[str, ...], tipos: dict[str, str]
) -> dict[str, str]:
    """Regras de qualidade da entidade: {nome: condição SQL verdadeira para o registro válido}."""
    regras = {}
    for coluna in (*chaves, *obrigatorias):
        regras[f"{coluna}_nao_nulo"] = f"{coluna} IS NOT NULL"
    for coluna in tipos:
        regras[f"{coluna}_conversivel"] = f"{coluna} IS NOT NULL OR {coluna}_bruto IS NULL"
    regras["sem_linha_corrompida"] = "_corrupt_record IS NULL"
    regras["sem_coluna_inesperada"] = "_rescued_data IS NULL"
    return regras


def condicao_valida(regras: dict[str, str]) -> Column:
    """Verdadeira só quando todas as regras são verdadeiras (nulo conta como reprovado)."""
    condicao = F.lit(True)
    for expressao in regras.values():
        condicao = condicao & F.coalesce(F.expr(expressao), F.lit(False))
    return condicao


def motivos_reprovacao(regras: dict[str, str]) -> Column:
    """Nomes das regras violadas, separados por vírgula."""
    return F.concat_ws(
        ", ",
        *[
            F.when(~F.coalesce(F.expr(expressao), F.lit(False)), F.lit(nome))
            for nome, expressao in regras.items()
        ],
    )


def isolar_reprovados(df: DataFrame, regras: dict[str, str]) -> DataFrame:
    """Só os registros que violam alguma regra, com o motivo e o momento do isolamento."""
    return (
        df.where(~condicao_valida(regras))
        .withColumn("motivo", motivos_reprovacao(regras))
        .withColumn("_quarentena_em", F.current_timestamp())
    )

## Descrição das entidades

Cada entidade é descrita por dados, não por código: nome, tipos, chave, colunas obrigatórias e a coluna que ordena as versões. Acrescentar uma entidade nova = acrescentar uma linha.

| Campo | Significado |
|---|---|
| `tipos` | colunas convertidas de `string` e o tipo de destino (as demais ficam como estão) |
| `chaves` | colunas que identificam o registro; nunca nulas |
| `obrigatorias` | outras colunas que não podem ser nulas |
| `sequence_by` | coluna que decide qual versão vale quando a chave se repete. **Vazio** = tabela de referência: só tipada e validada, sem deduplicação por chave |

**`sequence_by` é sempre o tempo do negócio, nunca o `sim_date`** (desenho S3). Exceção: `order_payments` não tem nenhum timestamp, então usa `sim_date` como critério de desempate. Isso é seguro porque as duplicatas de pagamento são cópias idênticas; a exceção entra no ADR da fase.

**Campos da etapa 2.3**

| Campo | Significado |
|---|---|
| `regras_extras` | regras de domínio, no mesmo formato de `montar_regras`; quem violar é descartado e vai para a quarentena com o nome da regra |
| `avisos` | regras que só **medem** (`expect`): o registro continua na silver e a contagem de falhas aparece no log de eventos. Servem para campos opcionais, como `payment_gateway` |
| `correcoes` | lista de `Correcao`: formato reconhecido, troca a aplicar e nome gravado em `_dq_correcao` (decisão D1: corrigir e marcar, em vez de mandar para a quarentena) |

Decisão D1: um preço `59,90` é uma venda real; mandá-lo para a quarentena tiraria receita do GMV por um problema de formato. A correção só vale para o padrão exato `número,dígitos`; qualquer outro formato continua reprovado.

In [0]:
# O QUE FAZ: declara a descrição de cada entidade (incluindo regras de domínio, avisos e correções
#   da etapa 2.3 e os feeds de mudança da 2.5), os domínios conhecidos e a lista das 10 entidades.
#   Nenhum dado é lido aqui.
#
# ETAPA 2.5 (customers e sellers):
# - sequence_by="change_sequence": a versão 1 (INSERT), 2 (UPDATE) ou 3 (DELETE) mais alta da chave
#   vence. Duplicatas e reenvios têm a mesma versão e o mesmo conteúdo, então o resultado é o mesmo.
# - "operation = 'DELETE' OR (...)": a linha DELETE chega com cidade, UF e CEP nulos; a regra só
#   exige o cadastro completo nas outras operações. OR em SQL: basta um lado ser verdadeiro.
# - O CEP não é tipado como número: "01310" virar 1310 perderia o zero à esquerda.
#
# RECURSOS:
# - @dataclass(frozen=True): cria uma classe só de dados, imutável (não pode ser alterada depois).
# - tuple (...): como uma lista que não muda; serve para uma configuração fixa. Uma tupla de um
#   item leva vírgula no final: ("order_id",).
# - dict {coluna: "TIPO"}: tipos SQL aceitos por try_cast (INT, DOUBLE, DATE, TIMESTAMP, DECIMAL(p,s)).
# - DECIMAL(12,2): número com até 12 dígitos, 2 depois da vírgula. Usado para dinheiro, porque
#   DOUBLE acumula erro de arredondamento na soma (GMV).
# - sequence_by = None: tabela de referência (sem deduplicação por chave).
# - field(default_factory=dict): valor padrão de um campo mutável (cada entidade ganha o seu
#   dicionário vazio; um dict direto como padrão seria compartilhado e o Python recusa).
# - r"...": texto "cru"; a barra invertida de uma expressão regular não é interpretada pelo Python.
# - ^\d+,\d{1,2}$ : regex = começa (^) com um ou mais dígitos (\d+), vírgula, 1 ou 2 dígitos
#   (\d{1,2}) e termina ($).
# - IN ('a','b'): teste de pertencimento a uma lista, em SQL.
# - BETWEEN a AND b: valor entre a e b, inclusive.
@dataclass(frozen=True)
class Correcao:
    coluna: str
    nome: str
    padrao: str
    de: str
    para: str


@dataclass(frozen=True)
class Entidade:
    nome: str
    descricao: str
    tipos: dict
    chaves: tuple
    obrigatorias: tuple = ()
    sequence_by: str | None = None
    regras_extras: dict = field(default_factory=dict)
    avisos: dict = field(default_factory=dict)
    correcoes: tuple = ()


STATUS_CONHECIDOS = (
    "created", "approved", "invoiced", "processing",
    "shipped", "delivered", "canceled", "unavailable",
)
TIPOS_PAGAMENTO = ("credit_card", "boleto", "voucher", "debit_card", "not_defined")
OPERACOES_CDC = "('INSERT', 'UPDATE', 'DELETE')"


ENTIDADES = (
    Entidade(
        nome="orders",
        descricao="Pedidos (cabeçalho), um por order_id.",
        tipos={
            "order_purchase_timestamp": "TIMESTAMP",
            "order_estimated_delivery_date": "TIMESTAMP",
            "sim_date": "DATE",
        },
        chaves=("order_id",),
        obrigatorias=("customer_id", "order_purchase_timestamp", "order_estimated_delivery_date"),
        sequence_by="order_purchase_timestamp",
    ),
    Entidade(
        nome="order_items",
        descricao="Itens de pedido, um por (order_id, order_item_id).",
        tipos={
            "order_item_id": "INT",
            "shipping_limit_date": "TIMESTAMP",
            "price": "DECIMAL(12,2)",
            "freight_value": "DECIMAL(12,2)",
            "sim_date": "DATE",
        },
        chaves=("order_id", "order_item_id"),
        obrigatorias=("product_id", "seller_id", "shipping_limit_date", "price", "freight_value"),
        sequence_by="shipping_limit_date",
        regras_extras={
            "item_id_positivo": "order_item_id >= 1",
            "preco_nao_negativo": "price >= 0",
            "frete_nao_negativo": "freight_value >= 0",
        },
        correcoes=(
            Correcao(
                coluna="price",
                nome="price_virgula_decimal",
                padrao=r"^\d+,\d{1,2}$",
                de=",",
                para=".",
            ),
        ),
    ),
    Entidade(
        nome="order_payments",
        descricao="Pagamentos de pedido, um por (order_id, payment_sequential).",
        tipos={
            "payment_sequential": "INT",
            "payment_installments": "INT",
            "payment_value": "DECIMAL(12,2)",
            "sim_date": "DATE",
        },
        chaves=("order_id", "payment_sequential"),
        obrigatorias=("payment_type", "payment_value"),
        sequence_by="sim_date",
        regras_extras={
            "valor_pagamento_nao_negativo": "payment_value >= 0",
            "parcelas_nao_negativas": "payment_installments IS NULL OR payment_installments >= 0",
            "tipo_pagamento_conhecido": "payment_type IN (" + ", ".join(f"'{t}'" for t in TIPOS_PAGAMENTO) + ")",
        },
        avisos={"payment_gateway_preenchido": "payment_gateway IS NOT NULL"},
    ),
    Entidade(
        nome="order_reviews",
        descricao="Avaliações de pedido, uma por (review_id, order_id).",
        tipos={
            "review_score": "INT",
            "review_creation_date": "TIMESTAMP",
            "review_answer_timestamp": "TIMESTAMP",
            "sim_date": "DATE",
        },
        chaves=("review_id", "order_id"),
        obrigatorias=("review_score", "review_answer_timestamp"),
        sequence_by="review_answer_timestamp",
        regras_extras={"nota_entre_1_e_5": "review_score BETWEEN 1 AND 5"},
    ),
    Entidade(
        nome="order_status_events",
        descricao="Eventos de status do pedido, um por (order_id, event_sequence).",
        tipos={
            "event_sequence": "INT",
            "event_ts": "TIMESTAMP",
            "sim_date": "DATE",
        },
        chaves=("order_id", "event_sequence"),
        obrigatorias=("status", "event_ts"),
        sequence_by="event_ts",
        regras_extras={
            "status_conhecido": "status IN (" + ", ".join(f"'{t}'" for t in STATUS_CONHECIDOS) + ")",
        },
    ),
    Entidade(
        nome="products",
        descricao="Cadastro de produtos (referência, carga única).",
        tipos={
            "product_name_lenght": "INT",
            "product_description_lenght": "INT",
            "product_photos_qty": "INT",
            "product_weight_g": "INT",
            "product_length_cm": "INT",
            "product_height_cm": "INT",
            "product_width_cm": "INT",
            "sim_date": "DATE",
        },
        chaves=("product_id",),
        regras_extras={
            "medidas_nao_negativas": "coalesce(product_weight_g, 0) >= 0 AND coalesce(product_length_cm, 0) >= 0 AND coalesce(product_height_cm, 0) >= 0 AND coalesce(product_width_cm, 0) >= 0",
        },
    ),
    Entidade(
        nome="geolocation",
        descricao="Coordenadas por prefixo de CEP (referência, carga única). Sem chave única.",
        tipos={
            "geolocation_lat": "DOUBLE",
            "geolocation_lng": "DOUBLE",
            "sim_date": "DATE",
        },
        chaves=(),
        obrigatorias=("geolocation_zip_code_prefix", "geolocation_lat", "geolocation_lng"),
        regras_extras={
            "latitude_valida": "geolocation_lat BETWEEN -90 AND 90",
            "longitude_valida": "geolocation_lng BETWEEN -180 AND 180",
        },
    ),
    Entidade(
        nome="category_translation",
        descricao="Tradução do nome da categoria de produto (referência, carga única).",
        tipos={"sim_date": "DATE"},
        chaves=("product_category_name",),
        obrigatorias=("product_category_name_english",),
    ),
    # Etapa 2.5: feeds de mudança. sequence_by = change_sequence (1 INSERT, 2 UPDATE, 3 DELETE):
    # a maior versão da chave vence. A linha DELETE chega com todos os atributos nulos, por isso o
    # cadastro só é exigido quando a operação não é DELETE. O CEP continua texto (zeros à esquerda).
    Entidade(
        nome="customers",
        descricao="Clientes (feed de mudanças), uma linha por customer_id: a última versão. Contém dados pessoais.",
        tipos={"change_sequence": "INT", "change_ts": "TIMESTAMP", "sim_date": "DATE"},
        chaves=("customer_id",),
        obrigatorias=("operation", "change_sequence", "change_ts"),
        sequence_by="change_sequence",
        regras_extras={
            "operacao_conhecida": "operation IN " + OPERACOES_CDC,
            "cadastro_completo_exceto_delete": "operation = 'DELETE' OR (customer_unique_id IS NOT NULL AND customer_city IS NOT NULL AND customer_state IS NOT NULL AND customer_zip_code_prefix IS NOT NULL)",
        },
    ),
    Entidade(
        nome="sellers",
        descricao="Vendedores (feed de mudanças), uma linha por seller_id: a última versão.",
        tipos={"change_sequence": "INT", "change_ts": "TIMESTAMP", "sim_date": "DATE"},
        chaves=("seller_id",),
        obrigatorias=("operation", "change_sequence", "change_ts"),
        sequence_by="change_sequence",
        regras_extras={
            "operacao_conhecida": "operation IN " + OPERACOES_CDC,
            "cadastro_completo_exceto_delete": "operation = 'DELETE' OR (seller_city IS NOT NULL AND seller_state IS NOT NULL AND seller_zip_code_prefix IS NOT NULL)",
        },
    ),
)

## Registro das tabelas no pipeline

`registrar_entidade` declara, para uma entidade, as peças do fluxo:

1. **`<e>_tipado` (view):** lê a bronze como *stream*, aplica `tipar` e `corrigir_formato`. É uma view: não ocupa armazenamento, só dá nome ao passo.
2. **`<e>_validos` (view com expectations):** `expect_all_or_drop` descarta quem violar qualquer regra e **registra a contagem por regra** no log de eventos do pipeline (base da etapa 2.4). Os `avisos` entram com `expect_all`: só medem, não descartam.
3. **`quality.<e>_quarentena` (tabela):** os mesmos reprovados, com o motivo. O `drop` sozinho faria o registro sumir sem rastro.
4. **`silver.<e>`:**
   - com `sequence_by`: `create_auto_cdc_flow` SCD tipo 1 mantém **uma linha por chave**, e vence a versão de maior `sequence_by` (trata duplicata, chegada atrasada e fora de ordem). É idempotente: rodar de novo sem dados novos não muda nada. O histórico fica para a Fase 3;
   - sem `sequence_by` (referência): tabela de streaming simples com os registros válidos.

*Streaming* aqui significa leitura incremental: cada execução processa só as linhas novas da bronze e guarda o ponto onde parou num checkpoint gerenciado pelo pipeline.

In [0]:
# O QUE FAZ: declara a função registrar_entidade e a chama para as 10 entidades. É aqui que o
#   grafo do pipeline nasce: cada chamada cria 2 views, a tabela de quarentena e a tabela silver.
#   O código das funções internas só roda quando o pipeline é executado.
#
# DECORADORES E FUNÇÕES DO PIPELINE:
# - @dp.view(name=..., comment=...): define uma view do pipeline (resultado intermediário, sem
#   gravar tabela). O nome é como as outras definições a chamam.
# - @dp.table(name=..., comment=...): define uma tabela gerenciada, mantida pelo pipeline. Quando a
#   função devolve um DataFrame de streaming, vira uma streaming table (incremental). O nome
#   com três partes (catálogo.schema.tabela) grava em outro schema que não o padrão.
# - @dp.expect_all_or_drop(dicionário): aplica todas as regras; o registro que violar QUALQUER uma
#   é descartado e a contagem por regra vai para o log de eventos do pipeline.
# - @dp.expect_all(dicionário): aplica as regras só para MEDIR; o registro fica, e a contagem de
#   falhas por regra vai para o log de eventos.
# - dp.create_streaming_table(name, comment): declara a streaming table de destino do AUTO CDC.
# - dp.create_auto_cdc_flow(target, source, keys, sequence_by, stored_as_scd_type, ...): aplica as
#   linhas de "source" em "target" por chave (upsert), resolvendo duplicatas e chegada fora de
#   ordem pela coluna sequence_by.
#     * keys: colunas que identificam o registro.
#     * sequence_by: coluna que ordena as versões; vence a de maior valor.
#     * stored_as_scd_type=1: sobrescreve a versão antiga (sem histórico).
#     * except_column_list: colunas de "source" que NÃO vão para o destino.
#
# FUNÇÕES NATIVAS:
# - spark.readStream.table(nome): lê uma tabela como stream (só linhas novas a cada execução). Com
#   um nome simples ("orders_tipado"), lê o dataset do próprio pipeline.
# - DataFrame.drop(*colunas): remove colunas. O "*" desempacota a lista em argumentos separados.
# - F.col(nome): referência à coluna usada em sequence_by.
# - list comprehension [x for ...]: monta uma lista em uma linha.
def registrar_entidade(e: Entidade) -> None:
    bronze = f"{CATALOG}.bronze.{e.nome}"
    silver = f"{CATALOG}.silver.{e.nome}"
    quarentena = f"{CATALOG}.quality.{e.nome}_quarentena"
    tipado, validos = f"{e.nome}_tipado", f"{e.nome}_validos"

    regras = {**montar_regras(e.chaves, e.obrigatorias, e.tipos), **e.regras_extras}
    # colunas usadas só pelas regras e pela quarentena: não vão para a silver
    fora_da_silver = [f"{c}_bruto" for c in e.tipos] + ["_corrupt_record", "_rescued_data"]

    @dp.view(name=tipado, comment=f"{e.nome} da bronze com os tipos convertidos; falha vira nulo.")
    def _tipado():
        return corrigir_formato(tipar(spark.readStream.table(bronze), e.tipos), e.correcoes, e.tipos)

    def _validos():
        return spark.readStream.table(tipado)

    # Decoradores aplicados à mão: assim os avisos só entram quando a entidade tem algum
    # (expect_all com dicionário vazio não tem utilidade). Equivale a empilhar @dp.view,
    # @dp.expect_all e @dp.expect_all_or_drop sobre a função.
    _validos = dp.expect_all_or_drop(regras)(_validos)
    if e.avisos:
        _validos = dp.expect_all(e.avisos)(_validos)
    dp.view(name=validos, comment=f"{e.nome} que cumpre todas as regras de qualidade.")(_validos)

    @dp.table(
        name=quarentena,
        comment=f"{e.nome} reprovado em alguma regra de qualidade, com o motivo. Não entra na silver.",
    )
    def _quarentena():
        return isolar_reprovados(spark.readStream.table(tipado), regras)

    if e.sequence_by:
        dp.create_streaming_table(
            name=silver,
            comment=f"{e.descricao} Tipada e deduplicada. Fonte: {bronze}.",
        )
        dp.create_auto_cdc_flow(
            target=silver,
            source=validos,
            keys=list(e.chaves),
            sequence_by=F.col(e.sequence_by),
            stored_as_scd_type=1,
            except_column_list=fora_da_silver,
        )
    else:

        @dp.table(name=silver, comment=f"{e.descricao} Tipada. Fonte: {bronze}.")
        def _silver():
            return spark.readStream.table(validos).drop(*fora_da_silver)


for entidade in ENTIDADES:
    registrar_entidade(entidade)

## Como executar e conferir

Este bloco é só roteiro, não é código do pipeline.

**Etapa 2.5:** `customers` e `sellers` são tabelas novas, sem estado anterior. Basta *Dry run* e depois uma execução **normal** (*Run pipeline*): só elas são processadas, e as outras 8 entidades não mudam (nenhum dado novo na bronze). Não use *Full refresh* agora.

**Etapa 2.3 (já feita), executar com *Full refresh all*** (Run pipeline ▾ → *Full refresh all*), e não com a execução normal:

- um streaming não relê o que já processou, então os 4 itens de `price` com vírgula, que a 2.2 mandou para a quarentena, não voltariam para a silver, e as regras novas não seriam aplicadas ao histórico;
- o *full refresh* apaga e recria **só as tabelas do pipeline** (silver e quarentena); a bronze e os checkpoints dela não são tocados;
- o histórico todo (cerca de 1,9 milhão de linhas) é reprocessado em poucos minutos.

Depois, rode uma execução normal para provar a idempotência.

**Conferir (no SQL editor, não neste arquivo).** Esperado:

```sql
-- 1. os 4 itens corrigidos voltaram para a silver, marcados
SELECT count(*) AS itens, count(DISTINCT order_id, order_item_id) AS distintos FROM olist.silver.order_items;            -- 111.592
SELECT order_id, order_item_id, price, _dq_correcao FROM olist.silver.order_items WHERE _dq_correcao IS NOT NULL;         -- 4 linhas: 59.90, 30.97, 24.99, 77.90

-- 2. a quarentena de order_items só tem a linha corrompida
SELECT motivo, count(*) FROM olist.quality.order_items_quarentena GROUP BY motivo;                                       -- 1 linha: sem_linha_corrompida (+ chaves nulas)

-- 3. as demais quarentenas continuam vazias (nenhum dado atual viola as regras de domínio)
SELECT 'orders' t, count(*) FROM olist.quality.orders_quarentena
UNION ALL SELECT 'order_payments', count(*) FROM olist.quality.order_payments_quarentena
UNION ALL SELECT 'order_reviews', count(*) FROM olist.quality.order_reviews_quarentena
UNION ALL SELECT 'order_status_events', count(*) FROM olist.quality.order_status_events_quarentena
UNION ALL SELECT 'products', count(*) FROM olist.quality.products_quarentena
UNION ALL SELECT 'geolocation', count(*) FROM olist.quality.geolocation_quarentena;                                      -- todas 0

-- 4. as demais silvers não mudam: order_payments 102.906 · order_reviews 95.253 · order_status_events 387.691

-- 5. etapa 2.5: uma linha por chave, com a última versão
SELECT count(*) AS linhas, count(DISTINCT customer_id) AS chaves FROM olist.silver.customers;                            -- 98.476 · 98.476
SELECT count(*) AS linhas, count(DISTINCT seller_id)   AS chaves FROM olist.silver.sellers;                              -- 3.095 · 3.095
SELECT operation, count(*) FROM olist.silver.customers GROUP BY operation;                                               -- DELETE 288 (o resto INSERT/UPDATE)
SELECT operation, count(*) FROM olist.silver.sellers   GROUP BY operation;                                               -- DELETE 21
SELECT count(*) FROM olist.quality.customers_quarentena;                                                                 -- 0
SELECT count(*) FROM olist.quality.sellers_quarentena;                                                                   -- 0
```

As linhas `DELETE` ficam na silver, com a cidade, a UF e o CEP nulos: o feed já manda o `DELETE` sem os atributos, e o AUTO CDC SCD1 troca a versão antiga pela nova. A decisão sobre o que fazer com elas (SCD2 ou exclusão de verdade) é da Fase 3 e da Fase 6.

**Aviso `payment_gateway_preenchido`:** não aparece nas tabelas. Veja em *Pipeline → clique em `order_payments_validos` → aba *Data quality*: 133 registros com falha (gateway nulo antes de 2026-10-08), todos mantidos na silver.

**Idempotência:** execute o pipeline de novo (normal) e repita a consulta 1; nenhuma contagem pode mudar.